# Tree structure
Nearest neighbor search (NNS) on a point cloud is often the most time-consuming part of point cloud processing. A naive NNS computes the distance between a query and **every** point, so its cost grows in proportion to the number of points.

A *spatial tree structure* accelerates NNS by recursively dividing the 3D space into smaller cells. During a search we can skip (prune) cells that are too far from the query, so we no longer need to look at every point.

This section introduces the **octree**, one of the most common tree structures for point clouds.
- Build an octree from a point cloud
- Visualize how the space is divided
- k nearest neighbor (kNN) search with the octree
- Radius nearest neighbor search with the octree

In [ ]:
%load_ext autoreload
%autoreload 2

## What is an octree?
An octree starts from a single cube (the *root*) that contains the whole point cloud. Each cube is divided into **8 equal sub-cubes (octants)** along the x, y and z axes through its center. Each octant is divided again, and so on.

The division stops when a cube reaches a maximum depth, or when it contains few enough points (here, `min_points` or less). Such a final cube is called a *leaf* and it stores the points inside it.

Because each cube knows the region of space it covers, a search can compute the distance from the query to a cube and ignore the whole cube (and all its points) when it is too far away.

This section uses the following code:

In [ ]:
# for octree
import numpy as np
from tutlibs.tree import Octree, boundaries_to_lines

# for description
from tutlibs.io import Points as io
from tutlibs.utils import single_color
from tutlibs.visualization import JupyterVisualizer as jv
import inspect

## Build an octree and visualize the cells
First, we build an octree from a point cloud and draw the boundary of every leaf cube. Cubes are small where points are dense and large where points are sparse, because dense regions are divided more times.

In [ ]:
# load a point cloud data.
coords, _, _ = io.read("../data/bunny_pc.ply")

# build an octree.
# max_depth: maximum number of subdivisions.
# min_points: a cube becomes a leaf when it has this number of points or less.
tree = Octree(coords, max_depth=7, min_points=16)

# get the cube boundary of every leaf and convert it to cube edges.
min_corners, max_corners = tree.leaf_boundaries()
lines = boundaries_to_lines(min_corners, max_corners)
print("number of points:", len(coords))
print("number of leaf cells:", len(min_corners))

# visualize the point cloud (blue) and the leaf cubes (green).
obj_points = jv.point(coords, single_color("#0000ff", len(coords)))
obj_lines = jv.line(lines, single_color("#00ff00", len(lines)))
jv.display([obj_points, obj_lines])

The figure above shows that the octree partitions the space into cubes that adapt to the point density.

Next, we look at how the octree is built. The `Octree` constructor computes a cubic root region that contains all points and then recursively divides it.

In [ ]:
print(inspect.getsource(Octree._build))

The `_build` process is as follows:
1. If the node reached `max_depth` or holds `min_points` points or less, make it a leaf that stores the point indices.
2. Otherwise, assign each point to one of the 8 octants. The octant is encoded as a 3 bit code: bit 0 is the x axis, bit 1 is the y axis and bit 2 is the z axis (1 when the coordinate is larger than the cube center).
3. Recursively build a child node for every non-empty octant.

## kNN search with the octree
kNN finds the $k$ nearest points to a query. With the octree, we visit cubes from the closest to the farthest and keep the current $k$ best candidates. When the closest possible point of a cube is farther than the current $k$-th neighbor, the whole cube is pruned.

In [ ]:
# define queries.
queries = coords[::500]

# kNN with the octree (the octree searches one query at a time).
k = 10
nn_coords = []
for query in queries:
    idx, dist = tree.knn_search(query, k)
    nn_coords.append(coords[idx])
nn_coords = np.concatenate(nn_coords, axis=0)

# visualize results.
obj_points = jv.point(coords, single_color("#0000ff", len(coords)))
obj_nn_points = jv.point(nn_coords, single_color("#00ff00", len(nn_coords)))
obj_queries = jv.point(queries, single_color("#ff0000", len(queries)))
jv.display([obj_points, obj_nn_points, obj_queries])

The output shows $k$ neighbors (green) for each query (red). Let us look at the implementation.

In [ ]:
print(inspect.getsource(Octree._knn_search))

The `_knn_search` process is as follows:
1. Compute the minimum distance from the query to the cube. If we already have $k$ candidates and this distance is larger than the current $k$-th neighbor, prune the cube.
2. For a leaf, update a max-heap of size $k$ with the points in the cube (the heap always keeps the $k$ closest points found so far).
3. For an internal node, visit the children from the closest to the farthest so that pruning becomes more effective.

We can confirm that the octree kNN returns the same result as the naive (brute force) kNN.

In [ ]:
from tutlibs.nns import k_nearest_neighbors

query = queries[0:1]
bf_idx, bf_square_dist = k_nearest_neighbors(query, coords, k)
tree_idx, tree_dist = tree.knn_search(query[0], k)

print("brute force distances:", np.sqrt(bf_square_dist[0]))
print("octree distances     :", tree_dist)
print("same neighbors:", np.array_equal(np.sort(bf_idx[0]), np.sort(tree_idx)))

## Radius nearest neighbor search with the octree
Radius search finds every point within a radius $r$ from a query. The octree prunes any cube that does not overlap with the query sphere.

In [ ]:
# radius search with the octree.
r = 0.05
nn_coords = []
for query in queries:
    idx, dist = tree.radius_search(query, r)
    nn_coords.append(coords[idx])
nn_coords = np.concatenate(nn_coords, axis=0)

# visualize results.
obj_points = jv.point(coords, single_color("#0000ff", len(coords)))
obj_nn_points = jv.point(nn_coords, single_color("#00ff00", len(nn_coords)))
obj_queries = jv.point(queries, single_color("#ff0000", len(queries)))
jv.display([obj_points, obj_nn_points, obj_queries])

In [ ]:
print(inspect.getsource(Octree._radius_search))

The `_radius_search` process is as follows:
1. Compute the minimum distance from the query to the cube. If it is larger than $r$, the cube cannot contain neighbors, so prune it.
2. For a leaf, add the points whose distance to the query is within $r$.
3. For an internal node, search every child recursively.

## Summary
- A tree structure divides 3D space into cells so that NNS can prune far cells instead of checking every point.
- An octree recursively divides a cube into 8 octants until each cell is small enough.
- The same octree supports both kNN and radius search, and returns the same neighbors as the naive search.

Other well known tree structures for NNS are the **kd-tree** and the **ball tree**. They use the same idea of pruning far regions, but split the space in a different way.